# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}:\n{metadata.description}\n")
print(f"Cite As: {metadata.citeAs}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print an overview of all record sets and their associated fields/columns from the dataset metadata, referring to each by its `@id`.

In [ ]:
# Retrieve record set definitions
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

overview = {}
for record_set in record_sets:
    print(f"--- RecordSet: {record_set.name} (@id={record_set.id}) ---")
    field_ids = []
    for field in record_set.fields:
        field_ids.append(field.id)
        print(f"  Field: {field.name} (@id={field.id}) | Type: {field.data_type}")
    overview[record_set.id] = field_ids
    print()
# For demonstration, select the first recordset @id for further steps
main_record_set_id = record_sets[0].id
first_fields = overview[main_record_set_id][:3]
print(f"Example RecordSet id: {main_record_set_id}")
print(f"Example Field ids: {first_fields}")

### Preview Records from Main RecordSet
Let's preview a few records from the selected record set using its `@id`.

In [ ]:
# Preview a few records from the main record set using its @id
print(f"Example records from record set (@id={main_record_set_id}):\n")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(json.dumps(rec, indent=2))
    if i >= 2:
        break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(recs)
    dataframes[rs_id] = df
    print(f"Loaded record set: {rs_id} (shape: {df.shape})")

# Display columns of main record set
print(f"\nColumns for main record set (@id={main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical processing: select a numeric field, filter, normalize, and group. All fields are referenced using their `@id`.

For demonstration, we'll find a numeric field—such as `gad_7_total_score` if present (substitute with an actual field `@id` from the overview above), filter for high values, normalize, and then group by gender or a similar categorical field.

In [ ]:
# Choose main DataFrame and select fields
df = dataframes[main_record_set_id]

# Set the numeric and group field ids (replace with actual @ids as printed above)
field_candidates = list(df.columns)
# We'll heuristically select 'gad_7_total_score' if available; adjust name as needed
numeric_field_id = None
for f in field_candidates:
    if 'gad' in f and 'score' in f:
        numeric_field_id = f
        break
if not numeric_field_id:
    numeric_field_id = field_candidates[-1]  # fallback, pick the last column

# Similarly pick a group field
group_field_id = None
for f in field_candidates:
    if 'gender' in f.lower():
        group_field_id = f
        break
if not group_field_id:
    group_field_id = field_candidates[0]  # fallback, pick the first column

print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group field selected (@id): {group_field_id}")

# Filter for high scores (above threshold, e.g., 10)
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a category (e.g., gender)
    if group_field_id in df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} by {group_field_id} for filtered observations:")
        print(grouped)
else:
    print(f"Selected numeric field '{numeric_field_id}' is not numeric. Please check your field selection.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll create a histogram of the selected numeric field and a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mental health survey data using the Croissant schema and explored its structure and content using their `@id` identifiers. We performed exploratory filtering and normalization of a key numeric score (e.g., GAD-7), visualized distributions, and demonstrated grouping by demographic variables. This demonstrates the power of the `mlcroissant` library for structured, FAIR data analysis workflows.